## Gradient Boosting en Clasificación (Log Loss)

En clasificación binaria, Gradient Boosting **no predice directamente la probabilidad**, sino una transformación:

$$
F(x) = \log \left( \frac{p(x)}{1 - p(x)} \right)
$$

Es decir:

👉 El modelo vive en **log-odds (logit)**

y la probabilidad se obtiene como:

$$
p(x) = \sigma(F(x)) = \frac{1}{1 + e^{-F(x)}}
$$

---

## Estructura del modelo

Al igual que en regresión:

$$
F_m(x) = F_{m-1}(x) + \alpha \cdot h_m(x)
$$

pero ahora:

- $F(x)$ = log-odds
- $h_m(x)$ = árbol que corrige errores
- $\alpha$ = learning rate

---

## Inicialización

Se empieza con una constante:

$$
F_0 = \log \left( \frac{\bar{y}}{1 - \bar{y}} \right)
$$

donde:

$$
\bar{y} = \frac{1}{n} \sum y_i
$$

👉 Es decir: el modelo empieza con el **log-odds de la tasa base**

---

## Paso clave: cálculo de residuales

En clasificación, los residuales **no son**:

$$
y - F(x)
$$

ni están en log-odds.

Son:

$$
r_i = y_i - p(x_i)
$$

donde:

$$
p(x_i) = \sigma(F(x_i))
$$

---

## Interpretación

👉 El residual mide:

- cuánto se equivocó la probabilidad
- no cuánto se equivocó el log-odds

---

## Qué aprende el árbol

Cada árbol $h_m(x)$ aprende:

$$
r_i = y_i - p(x_i)
$$

👉 Es decir:

> aprende el error en probabilidad

---

## Actualización

Después se actualiza:

$$
F(x) \leftarrow F(x) + \alpha \cdot h_m(x)
$$

👉 Ojo:

- la suma ocurre en **log-odds**
- pero el error que se aprende viene de **probabilidades**

---

## Flujo completo por iteración

1. Convertir predicción actual a probabilidad:

$$
p(x) = \sigma(F(x))
$$

2. Calcular residuales:

$$
r = y - p(x)
$$

3. Entrenar árbol sobre $r$

4. Actualizar:

$$
F(x) \leftarrow F(x) + \alpha \cdot h(x)
$$

---

## Modelo final

$$
F_M(x) = F_0 + \alpha \sum_{m=1}^M h_m(x)
$$

y la probabilidad final es:

$$
p(x) = \sigma(F_M(x))
$$

---

## Comparación con regresión

| Elemento | Regresión (MSE) | Clasificación (Log Loss) |
|----------|----------------|--------------------------|
| Output del modelo | $y$ | log-odds |
| Conversión final | no | sigmoide |
| Residual | $y - \hat y$ | $y - p$ |
| Qué aprende el árbol | error directo | error en probabilidad |
| Espacio donde se suma | $y$ | log-odds |

---

## Insight clave (el más importante)

> El modelo vive en log-odds, pero aprende errores en probabilidad.

---

## Conexión con gradiente

En general:

$$
r_i = - \frac{\partial L}{\partial F(x_i)}
$$

Para log loss:

$$
r_i = y_i - p(x_i)
$$

👉 Esto es lo que hace que Gradient Boosting sea realmente un método de optimización.

---

## Frase mental correcta

> Clasificación con GBT = suma de árboles en log-odds que corrigen errores de probabilidad

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.inspection import PartialDependenceDisplay


# --- Limpieza de datos según lo indicado ---
df = pd.read_csv('titanic.csv')
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df = pd.get_dummies(df)
del df['Sex_female']
df['Age'] = df['Age'].fillna(df.Age.mean())

# Variables predictoras y variable objetivo
X = df.drop('Survived', axis=1)
y = df['Survived']

# --- División en conjunto de entrenamiento y test (80%-20%) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [2]:
# Crea un gradient boosting
from sklearn.ensemble import GradientBoostingClassifier
model=GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)
# Predicciones
probabilidades=model.predict_proba(X_test)[:, 1]
# Evaluación
roc_auc_score(y_score=probabilidades, y_true=y_test)


np.float64(0.8247694334650856)

In [3]:
## Cross validation 
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

pipeline=Pipeline([
    ('gbt',GradientBoostingClassifier())
])
params={
    'gbt__max_depth':[2,3,4,5],
    'gbt__n_estimators':[30,50],
    'gbt__learning_rate':[0.1,0.05]                      
}
grid=GridSearchCV(estimator=pipeline, cv=3, param_grid=params, scoring='roc_auc')
grid.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('gbt', GradientBoostingClassifier())]),
             param_grid={'gbt__learning_rate': [0.1, 0.05],
                         'gbt__max_depth': [2, 3, 4, 5],
                         'gbt__n_estimators': [30, 50]},
             scoring='roc_auc')

In [5]:
# Mejor estimador 
best_grid_gbt = grid.best_estimator_.named_steps['gbt']

In [6]:
# AUC
prob=best_grid_gbt.predict_proba(X_test)[:, 1]
roc_auc_score(y_score=prob, y_true=y_test)

np.float64(0.8295125164690382)

In [ ]:
## Calibracion

In [ ]:
# Entendiendo el modelo 
